# 不计分实验：逻辑回归的梯度下降




## 目标
在本实验中，你将：
- 实现逻辑回归的梯度下降更新步骤。
- 一个使用循环的版本
- 可选地，一个使用矩阵的版本


## 数据集
让我们从与之前相同的数据集开始。


In [ ]:
import numpy as np

X = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1]).reshape(-1,1)

和之前一样，我们将用一个辅助函数来绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，标签为 $y=0$ 的数据点显示为黑色圆圈。


In [ ]:
from lab_utils import plot_data
import matplotlib.pyplot as plt

plot_data(X,y)

# 把两个坐标轴都设为 0-6
plt.axis([0, 6, 0, 6])
# 设置 y 轴标签
plt.ylabel('$x_1$')
# 设置 x 轴标签
plt.xlabel('$x_0$')

## 逻辑回归的梯度

 首先，你将实现梯度的非向量化版本。然后，你将实现向量化版本。


### 非向量化版本



回忆一下，梯度下降算法使用梯度计算：
$$\\begin{align*}& \\text{repeat until convergence:} \\; \\lbrace \\newline \\; & b := b -  \\alpha \\frac{\\partial J(\\mathbf{w},b)}{\\partial b} \\newline       \\; & w_j := w_j -  \\alpha \\frac{\\partial J(\\mathbf{w},b)}{\\partial w_j} \\tag{1}  \\; & \\text{for j := 0..n-1}\\newline & \\rbrace\\end{align*}$$


其中每次迭代对所有 $j$ 的 $w_j$ 同时更新，且
$$
\\frac{\\partial J(\\mathbf{w},b)}{\\partial b}  = \\frac{1}{m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - \\mathbf{y}^{(i)}) \\tag{2}
$$
$$
\\frac{\\partial J(\\mathbf{w},b)}{\\partial w_j}  = \\frac{1}{m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - \\mathbf{y}^{(i)})x_{j}^{(i)} \\tag{3}
$$

* m 是数据集中的训练样本数

    
*  $f_{\\mathbf{w},b}(x^{(i)})$ 是模型的预测，而 $y^{(i)}$ 是实际标签




* 对于上面给定数据集的逻辑回归模型，模型可以表示为：

    $f_{\\mathbf{w},b}(x) = g(w_0 + w_1x_1 + w_2x_2)$

    其中 $g(z)$ 是 sigmoid 函数：

    $g(z) = \\frac{1}{1+e^{-z}}$ 
    



我们已经为你实现了 `sigmoid` 函数，你只需导入并使用它，如下面的代码块所示。


In [ ]:
from lab_utils import sigmoid 

print(sigmoid(0))

### 使用循环的 compute_gradient
为所有 $w_j$ 和 $b$ 实现上面的公式 (2)、(3)。
实现方法有很多，你可以选择其他做法。下面概述其中一种：
- 初始化用于累加 dJdw 和 dJdb 的变量
- 遍历所有样本
    - 计算该样本的误差 $g(\\mathbf{x}^{(i)T}\\mathbf{w} + b) - \\mathbf{y}^{(i)}$
    - 把误差加到 dJdb（上面的公式 2）
    - 对该样本中的每个输入值 $x_{j}^{(i)}$
        - 把误差乘以输入 $x_{j}^{(i)}$，加到 dJdw 对应的元素上。
- 把 dJdb 和 dJdw 除以样本总数（m）


<details>
<summary>
    <font size='3', color='darkgreen'><b>提示</b></font>
</summary>


〖CODE0〗



In [ ]:
def compute_gradient_logistic_loop(X, y, w, b): 
    """计算线性回归的梯度
 
    参数：
      X : (array_like 形状 (m,n)) 变量，例如房屋面积
      y : (array_like 形状 (m,1)) 实际值
      w : (array_like 形状 (n,1)) 模型参数取值
      b : (scalar)                模型参数取值
    返回
      dJdw: (array_like 形状 (n,1)) 代价对参数 w 的梯度。
      dJdb: (scalar)                代价对参数 b 的梯度。"""
    m,n = X.shape
    dJdw = np.zeros((n,1))
    dJdb = 0.
    err  = 0.

    ### 在此处编写代码 ###
    ### 解答开始 ###
    for i in range(m):
        err = sigmoid(X[i] @ w + b)  - y[i]    
        for j in range(n):
            dJdw[j] = dJdw[j] + err * X[i][j]
        dJdb = dJdb + err
    dJdw = dJdw/m
    dJdb = dJdb/m
    ### 解答结束 ###
    ### 代码编写结束 ###
        
    return dJdb[0],dJdw  #对 dJdb 取索引以返回标量值

In [ ]:
w = np.array([2.,3.]).reshape(-1,1)
b = 1.
dJdb, dJdw = compute_gradient_logistic_loop(X, y, w, b)
print(f"dJdb, non-vectorized version: {dJdb}" )
print(f"dJdw, non-vectorized version: {dJdw.tolist()}" )

用下面的单元格检查你梯度函数的实现。


**预期输出**

``` 
dJdb, non-vectorized version: 0.49861806546328574
dJdw, non-vectorized version: [[0.498333393278696], [0.49883942983996693]]
```

### （可选）向量化版本

你现在将实现梯度函数的向量化版本。

梯度公式的向量化版本为

$$\\frac{\\partial \\mathbf{J_{w,b}}(\\mathbf{X,y})}{\\partial \\mathbf{b}}= \\frac{1}{m} sum(\\mathbf{f_{w,b}} - \\mathbf{y}) \\tag{4}$$ 


$$\\nabla_{\\mathbf{w}}\\mathbf{J} = \\frac{1}{m} \\mathbf{X^T}(\\mathbf{f} - \\mathbf{y}) \\tag{5}$$ 

其中

$$ \\mathbf{f_{w,b}} = g(\\mathbf{X}  \\mathbf{w})$$

和之前一样，$g$ 是 sigmoid 函数



**练习**

你将使用上面的公式完成向量化代价函数。如果遇到困难，可以参考提示。





**调试提示：** 向量化代码有时会比较棘手。一个常用的调试策略是用 size 函数打印出所操作矩阵的尺寸。例如，给定大小为 6 × 3 的数据矩阵 $\\mathbf{X}$（6 个样本、3 个特征）和维度为 3x1 的向量 $\\mathbf{w}$，你可以发现 $\\mathbf{Xw}$ 是合法的乘法运算，而 $\\mathbf{wX}$ 不是。


<details>
<summary>
    <font size='3', color='darkgreen'><b>提示</b></font>
</summary>


〖CODE0〗



In [ ]:
def compute_gradient_logistic_matrix(X, y, w, b): 
    """计算线性回归的梯度
 
    参数：
      X : (array_like 形状 (m,n)) 变量，例如房屋面积
      y : (array_like 形状 (m,1)) 实际值
      w : (array_like 形状 (n,1)) 模型参数取值
      b : (scalar )               模型参数取值
    返回
      dJdw: (array_like 形状 (n,1)) 代价对参数 w 的梯度。
      dJdb: (scalar)                代价对参数 b 的梯度。"""
    m,n = X.shape
    ### 在此处编写代码 ###
    ### 解答开始 ###
    f_wb =  sigmoid(X @ w + b)      ##None
    err  = f_wb - y                 ##None
    dJdw = (1/m) * (X.T @ err)      ##None
    dJdb = (1/m) * np.sum(err)      ##None
    ### 解答结束 ###
    ### 代码编写结束 ###
        
    return dJdb,dJdw

现在让我们检查这个函数的输出是否等价于上面非向量化实现的输出。


In [ ]:
w = np.array([2.,3.]).reshape(-1,1)
b = 1.
dJdb, dJdw = compute_gradient_logistic_loop(X, y, w, b)
print(f"dJdb, non-vectorized version: {dJdb}" )
print(f"dJdw, non-vectorized version: {dJdw.tolist()}" )
dJdb, dJdw = compute_gradient_logistic_matrix(X, y, w, b)
print(f"dJdb, vectorized version: {dJdb}" )
print(f"dJdw, vectorized version: {dJdw.tolist()}" )
#print("Gradients computed by matrix version: \n", compute_gradient_logistic_matrix(X, y, w, b, predict))

**预期输出** 

```
dJdb, non-vectorized version: 0.49861806546328574
dJdw, non-vectorized version: [[0.498333393278696], [0.49883942983996693]]
dJdb, vectorized version: 0.49861806546328574
dJdw, vectorized version: [[0.498333393278696], [0.4988394298399669]]
```
